# 🤗 02 — Inférence NLP avec Hugging Face

Trois modèles HF pour analyser nos avis :
- **Sentiment FR** : `tblard/tf-allocine`
- **Topic zero-shot** : `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
- **Résumé FR** : `plguillou/t5-base-fr-sum-cnndm`

⚠️ Nécessite : `pip install transformers torch sentencepiece`

In [ ]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv('../data/raw/customer_reviews.csv')
df.head(3)

## 1. Sentiment Analysis

In [ ]:
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='tblard/tf-allocine',
    tokenizer='tblard/tf-allocine'
)

# Test sur 5 avis
for i, row in df.head(5).iterrows():
    result = sentiment_pipe(row['review_text'])[0]
    print(f"Note: {row['rating']}★ | Pred: {result['label']} ({result['score']:.2%})")
    print(f"  → {row['review_text'][:80]}...\n")

In [ ]:
# Inférence batch sur tous les avis (plus rapide)
results = sentiment_pipe(df['review_text'].tolist(), batch_size=32, truncation=True)
df['sentiment_label'] = ['positive' if r['label'] == 'POSITIVE' else 'negative' for r in results]
df['sentiment_score'] = [r['score'] for r in results]
df[['rating', 'sentiment_label', 'sentiment_score']].head()

## 2. Topic Classification (Zero-Shot)

L'avantage du zero-shot : **pas besoin de dataset d'entraînement** ! On donne juste les labels possibles.

In [ ]:
topic_pipe = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
)

candidate_labels = ['livraison', 'qualité', 'service client', 'prix', 'application']

# Test sur 3 avis
for text in df['review_text'].head(3):
    result = topic_pipe(text, candidate_labels, multi_label=True)
    top_topics = [(l, s) for l, s in zip(result['labels'][:3], result['scores'][:3])]
    print(f'Texte: {text[:80]}...')
    print(f'Top topics: {top_topics}\n')

## 3. Summarization (sur avis longs)

In [ ]:
summary_pipe = pipeline(
    'summarization',
    model='plguillou/t5-base-fr-sum-cnndm'
)

# Avis le plus long
longest = df.loc[df['review_length'].idxmax(), 'review_text']
print(f'Original: {longest}')
summary = summary_pipe(longest, max_length=40, min_length=10, do_sample=False)[0]
print(f'\nRésumé: {summary["summary_text"]}')

## Sauvegarde du dataset enrichi

In [ ]:
df.to_csv('../data/processed/reviews_enriched.csv', index=False)
print('✓ Sauvegardé')